**Install Required Packages**

In [12]:
!pip install pandas --quiet
!pip install tabulate --quiet
!pip install pdfplumber --quiet
!pip install tqdm

!pip install pymupdf --upgrade --prefer-binary --only-binary :all:

**Define page-finder & table-extractor functions**

In [13]:
import pandas as pd
import fitz
import pdfplumber

def find_pages(pdf_path: str, keyword: str) -> list[int]:
    """
    Return 1-based page numbers where `keyword` appears in the page text.
    """
    pages = []
    with fitz.open(pdf_path) as doc:
        for i in range(doc.page_count):
            if keyword in doc[i].get_text():
                pages.append(i + 1)
    return pages


def extract_tables(
    pdf_path: str,
    pages: list[int],
    period: str,
    drop_type: str = "Spread") -> pd.DataFrame:
    """
    From the given pages, pull out every table row where
    df['Period']==period AND df['Type']!=drop_type.
    Returns one concatenated DataFrame (or empty if none).
    """
    dfs = []
    with pdfplumber.open(pdf_path) as pdf:
        for pnum in pages:
            t = pdf.pages[pnum - 1].extract_table()
            if not t:
                continue
            df = pd.DataFrame(t[1:], columns=t[0])
            if "Type" in df.columns:
                df["Type"] = df["Type"].str.strip()
                df = df[df["Type"] != drop_type]
            if "Period" in df.columns:
                df = df[df["Period"] == period]
            if not df.empty:
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

**Define a function to determine if "In the Money"**

In [14]:
def find_in_the_money(df: pd.DataFrame) -> pd.DataFrame:
    """
    Coerce 'Exp Value' to numeric, extract numeric threshold from 'Display Name',
    compute 'Flag' as 1 if Exp Value > Threshold (else 0), and drop the intermediate 'Threshold' column.
    """
    return (
        df
        .assign(
            **{'Exp Value': lambda df: pd.to_numeric(df['Exp Value'], errors='coerce')},
            Threshold=lambda df: (
                df['Display Name']
                  .str.extract(r'>\s*([\d.]+)', expand=False)
                  .pipe(pd.to_numeric, errors='coerce')
            ),
        )
        .assign(
            Flag=lambda df: (df['Exp Value'] > df['Threshold']).astype(int)
        )
        .drop(columns=['Threshold'])
    )

**Define the clean and rename function**

In [15]:
import pandas as pd

def clean_and_rename(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Keeps the first unique 'Display Name'
    - Drops columns: Name, Period, Display Name, Type, Buyer, Seller, Exp Time
    - Renames:
        Business Date  → Date
        Exp Value      → Strike
        Flag           → In the Money
    """
    return (
        df
        .drop_duplicates(subset=["Display Name"], keep="first")
        .drop(columns=["Type", "Buyer", "Seller", "Exp Time"])
        .drop(columns=["Period", "Display Name"])
        .rename(columns={
            "Business Date": "Date",
            "Exp Value": "Strike",
            "Flag": "In the Money",
        })
    )

**Define a function to substitue the Ticker for the name**

In [16]:
import pandas as pd
from pathlib import Path
from typing import Union

def add_ticker_from_mapping(
    df: pd.DataFrame,
    mapping_file: Union[str, Path]) -> pd.DataFrame:
    """
    Load a mapping CSV with columns 'Display Name' and 'Ticker',
    then add a 'Ticker' column to `df` by substring-matching 'Name' against each 'Display Name'.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame containing a 'Name' column.
    mapping_file : str or Path
        Path to CSV with mapping of 'Display Name' to 'Ticker'.

    Returns
    -------
    pd.DataFrame
        A new DataFrame with an added 'Ticker' column.
    """
    mapping_df = pd.read_csv(mapping_file)
    
    def lookup_ticker(name: str) -> pd.Series:
        if pd.isna(name):
            return pd.NA
        for desc, ticker in zip(mapping_df['Display Name'], mapping_df['Ticker']):
            if desc in name:
                return ticker
        return pd.NA

    result = df.copy()
    result['Ticker'] = result['Name'].apply(lookup_ticker)

    print(f"About to fill {result['Ticker'].isna().sum()} missing tickers.")
    result = result.fillna({'Ticker': 'UNKNOWN'})

    return result

**Define a function to save the final result to an S3 CSV**

In [17]:
import io
import boto3
import pandas as pd
from botocore.exceptions import ClientError

def upload_df_to_s3(
    df: pd.DataFrame,
    bucket: str,
    key: str,
    region: str = None) -> None:
    """
    Uploads a DataFrame to S3 as CSV. Verifies bucket existence first.
    
    Parameters
    ----------
    df : pd.DataFrame
    bucket : str
        Name of the S3 bucket (no leading/trailing spaces).
    key : str
        S3 object key, e.g. "historical/2025-06-28-results.csv"
    region : str, optional
        AWS region where the bucket resides.
    """
    bucket = bucket.strip()
    s3 = boto3.client('s3', region_name=region)

    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError as e:
        code = e.response['Error']['Code']
        msg = e.response['Error']['Message']
        raise RuntimeError(
            f"Could not access bucket '{bucket}' (region={region}): {msg} (code {code})"
        ) from e

    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)

    try:
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
        print(f"✅ Uploaded to s3://{bucket}/{key}")
    except ClientError as e:
        raise RuntimeError(
            f"Failed to upload CSV to s3://{bucket}/{key}: "
            f"{e.response['Error']['Message']}"
        ) from e

**Define function to avoid re-processing processed files**

In [18]:
import json
from botocore.exceptions import ClientError

def load_manifest(
    s3_client: boto3.client, bucket_name: str, manifest_key: str = "manifests/processed_files.json"
) -> set[str]:
    """
    Download and parse the JSON manifest of processed PDFs.
    Returns a set of keys, or empty set if it doesn’t exist.
    """
    try:
        resp = s3_client.get_object(Bucket=bucket_name, Key=manifest_key)
        return set(json.loads(resp["Body"].read()))
    except ClientError as e:
        # If the object isn’t found, return empty set; else re-raise
        if e.response["Error"]["Code"] == "NoSuchKey":
            return set()
        raise

def save_manifest(
    processed: set[str],
    s3_client: boto3.client,
    bucket_name: str,
    manifest_key: str = "manifests/processed_files.json"
) -> None:
    """
    Upload the updated manifest back to S3.
    """
    s3_client.put_object(
        Bucket=bucket_name,
        Key=manifest_key,
        Body=json.dumps(list(processed)).encode("utf-8"),
        ContentType="application/json",
    )

def upload_df_to_s3(
    df: pd.DataFrame,
    s3_client: boto3.client,
    bucket_name: str,
    key: str,
    region: str = "us-east-1",
) -> None:
    """
    Write a DataFrame to S3 as CSV.
    """
    csv_body = df.to_csv(index=False).encode("utf-8")
    s3_client.put_object(
        Bucket=bucket_name,
        Key=key,
        Body=csv_body,
        ContentType="text/csv",
    )

**Define a function to get all the files related to trading results**

In [19]:
from typing import List

def list_nadex_trading_results(bucket, prefix: str = "") -> List[str]:
    """
    List PDF keys in the given bucket under `prefix`
    that contain 'tradingResults' and end with .pdf.
    """
    return [
        obj.key
        for obj in bucket.objects.filter(Prefix=prefix)
        if "tradingResults" in obj.key and obj.key.lower().endswith(".pdf")
    ]


**Define functions to setup access to S3, and filter the list of pdf's returned**

In [20]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

from datetime import date, datetime
from pathlib import Path

from typing import Iterable, List, Dict

def create_s3_clients(
    profile: str = "default", region: str = "us-east-1"
) -> Dict[str, boto3.client]:
    session = boto3.Session(profile_name=profile, region_name=region)
    return {
        "public": session.client(
            "s3",
            config=Config(signature_version=UNSIGNED),
            region_name=region,
        ),
        "private": session.client("s3"),
        "resource": session.resource("s3"),
    }

def get_bucket(resource: boto3.resource, name: str):
    return resource.Bucket(name)

def parse_key_date(key: str) -> date:
    stem = Path(key).stem
    return datetime.strptime(stem[:8], "%Y%m%d").date()

def filter_new_pdfs(
    keys: Iterable[str],
    processed: Iterable[str],
    start: date = date(2024, 1, 1),
    end: date | None = None,
) -> List[str]:
    """
    Return keys between start & end (inclusive) that aren’t in processed.
    """
    if end is None:
        end = date.today()
    return [
        key
        for key in keys
        if start <= (d := parse_key_date(key)) <= end
        and key not in processed
    ]

**Define a function to run the Pipeline**
1. For each PDF in the folder:
2. Read the PDF file
3. Extract the Tables with Daily contracts
4. Create a CSV
5. Write the CSV to S3
6. Log the PDF file as 'processed

In [21]:
from pathlib import Path
from datetime import date
from typing import List

from tqdm import tqdm

def _process_pdf(
    pdf_key: str,
    target: str,
    mapping_file: Path,
    public_s3,
    private_s3,
    bucket_name: str,
    nadex_bucket_name: str,
    tmp_dir: Path = Path("/tmp"),
) -> None:
    """
    Download a single PDF, extract/transform, upload the CSV, and
    print a success message. Raises on any step failure.
    """
    local_pdf = tmp_dir / Path(pdf_key).name
    public_s3.download_file(
        Bucket=nadex_bucket_name,
        Key=pdf_key,
        Filename=str(local_pdf),
    )

    pages = find_pages(str(local_pdf), target)
    df = (
        extract_tables(str(local_pdf), pages, target)
        .pipe(find_in_the_money)
        .pipe(clean_and_rename)
        .pipe(add_ticker_from_mapping, mapping_file)
    )

    s3_key = f"historical/{Path(pdf_key).stem}.csv"
    upload_df_to_s3(df, private_s3, bucket_name, s3_key)

def run_nadex_pipeline(
    mapping_file: Path,
    target: str,
    bucket_name: str,
    nadex_bucket_name: str,
    start: date,
    end: date,
):
    """
    Orchestrates the full Nadex PDF → CSV pipeline over a given date range.

    Parameters:
    -----------
    mapping_file : Path to the CSV mapping file used to enrich extracted tables.
    target : The keyword to locate pages within each PDF (e.g. "Daily").
    bucket_name : Name of your own S3 bucket where resulting CSVs and the manifest are stored.
    nadex_bucket_name : Name of the public Nadex S3 bucket from which PDFs are downloaded (unsigned).
    start : Lower bound (inclusive) on PDF dates to process (parsed from filenames).
    end : Upper bound (inclusive) on PDF dates to process.

    Actions:
    --------
    1. Bootstraps three S3 interfaces:
       - `public_s3`: an unsigned client to download Nadex PDFs.
       - `private_s3`: a signed client for uploading CSVs & manifest.
       - `s3_resource`: resource interface to enumerate bucket objects.
    2. Constructs `buckets` dict with Bucket objects for both source (market) and destination.
    3. Loads the JSON “processed files” manifest from your private bucket into a `processed` set.
    4. Iterates over every PDF key in the Nadex (market) bucket:
       a. Skips any key already in `processed`, accumulating in `skipped`.
       b. Parses the date out of the filename and skips if outside `[start, end]`.
       c. Calls `_process_pdf(...)` to:
          • Download PDF locally,
          • Extract & transform tables,
          • Enrich with ticker mapping,
          • Upload the resulting CSV back to your bucket.
       d. On success, adds the key to `processed`; on exception, logs to `errors`.
    5. After the loop finishes, writes the updated `processed` manifest back to S3.
    6. Prints a summary of how many files were processed, skipped, and errored.

    """
    clients = create_s3_clients()
    public_s3 = clients["public"]
    private_s3 = clients["private"]
    s3_resource = clients["resource"]
    buckets = {
        "daily":  get_bucket(s3_resource, bucket_name),
        "market": get_bucket(s3_resource, nadex_bucket_name),
    }

    processed = load_manifest(private_s3, bucket_name)
    errors: Dict[str, str]  = {}

    # 1) List *all* PDFs in the Nadex bucket
    all_keys = list_nadex_trading_results(buckets["market"], prefix="")

    # 2) Filter those to your date window
    date_range_keys = [
        k for k in all_keys
        if start <= parse_key_date(k) <= end
    ]

    # 3) Split into new vs skipped-within-date-range
    new_keys = [k for k in date_range_keys if k not in processed]
    skipped  = [k for k in date_range_keys if k in processed]

    errors: Dict[str, str] = {}
    
    for pdf_key in tqdm(
        new_keys,
        desc="Processing PDFs",
        ascii=True,
    ):

    # ─── Main loop (single statement!) ────────────────────────────────────────
        try:
            _process_pdf(
                pdf_key,
                target,
                mapping_file,
                public_s3,
                private_s3,
                bucket_name,
                nadex_bucket_name,
            )
            processed.add(pdf_key)
        except Exception as exc:
            errors[pdf_key] = str(exc)

    save_manifest(processed, private_s3, bucket_name)

    print()
    print(f"Done — processed: {len(processed)} files")
    print(f"       skipped:   {len(skipped)} files already processed")
    print(f"       errors:    {len(errors)} failures")
    
    if errors:
        print("\n❗ Failed files and their errors:")
        for fname, err in errors.items():
            print(f"  • {fname}: {err}")

**Run the pipeline & display**

In [22]:
from pathlib import Path
from datetime import date

run_nadex_pipeline(
    mapping_file=Path("ticker_from_description.csv"),
    target="Daily",
    bucket_name="nadex-daily-results",
    nadex_bucket_name="market-data-prod.nadex.com",
    start=date(2025, 4, 1),
    end=date.today(),
)

Processing PDFs: 100%|##########| 33/33 [01:04<00:00,  1.95s/it]


Done — processed: 73 files
       skipped:   72 files already processed
       errors:    32 failures

❗ Failed files and their errors:
  • 20250405_tradingResults.pdf: 'Exp Value'
  • 20250406_tradingResults.pdf: 'Exp Value'
  • 20250412_tradingResults.pdf: 'Exp Value'
  • 20250413_tradingResults.pdf: 'Exp Value'
  • 20250418_tradingResults.pdf: 'Exp Value'
  • 20250419_tradingResults.pdf: 'Exp Value'
  • 20250420_tradingResults.pdf: 'Exp Value'
  • 20250426_tradingResults.pdf: 'Exp Value'
  • 20250427_tradingResults.pdf: 'Exp Value'
  • 20250503_tradingResults.pdf: 'Exp Value'
  • 20250504_tradingResults.pdf: 'Exp Value'
  • 20250510_tradingResults.pdf: 'Exp Value'
  • 20250511_tradingResults.pdf: 'Exp Value'
  • 20250517_tradingResults.pdf: 'Exp Value'
  • 20250518_tradingResults.pdf: 'Exp Value'
  • 20250524_tradingResults.pdf: 'Exp Value'
  • 20250525_tradingResults.pdf: 'Exp Value'
  • 20250531_tradingResults.pdf: 'Exp Value'
  • 20250601_tradingResults.pdf: 'Exp Value'
  • 2025